# Smoke Test — April 9 + Both April 20 Records

Runs all three record trainers back-to-back on a single Colab GPU using the same data.

**What each cell does:**
1. GPU check
2. Clone / update repo
3. Install dependencies + optional flash-attn
4. Download shared data — **5 train shards + val** (used by all three models)
5. Train + export: April 9 `LegalTTT`
6. Train + export: April 20 `AttnGate_MultiPhaseTTT_LaCT`
7. Train + export: April 20 `3LayerRecur_ParResid_AttnGate_MP4TTT`
8. Compare final metrics from all three logs

**Smoke-test settings (all three models share these):**
- `ITERATIONS=300` — fast check; not a quality run
- `TRAIN_BATCH_TOKENS=32768` — single-GPU batch (full record uses 786432 on 8×H100)
- `VAL_BATCH_TOKENS=32768` — reduced eval scope
- `MAX_WALLCLOCK_SECONDS=0` — no time cap; let all 300 steps finish
- `VAL_LOSS_EVERY=0` — skip periodic val; final only
- `GPTQ_CALIBRATION_BATCHES=8` — fast export check
- `PG_COLAB_DISABLE_COMPILE=1` — disables `torch.compile` via `sitecustomize.py`

**OOM recovery:** lower `GPTQ_CALIBRATION_BATCHES=4` or `TRAIN_BATCH_TOKENS=16384` in the failing cell only.

In [ ]:
import subprocess
print(subprocess.check_output(['nvidia-smi'], text=True))

In [ ]:
%%bash
set -euo pipefail
cd /content
if [[ ! -d parameter-golf/.git ]]; then
  git clone https://github.com/IanniMuliterno/parameter-golf.git parameter-golf
else
  cd parameter-golf
  git fetch origin
  git pull --ff-only || true
fi
echo "commit: $(git -C /content/parameter-golf rev-parse --short HEAD)"

In [ ]:
%%bash
set -euo pipefail
pip install --quiet sentencepiece huggingface_hub brotli
# flash-attn is optional; the stub in this colab dir provides a pure-PyTorch fallback
pip install --quiet "flash-attn>=2.6.0" \
    --extra-index-url https://download.pytorch.org/whl/cu124 || \
    echo "flash-attn install failed — fallback stub will be used for April 9 trainer."

In [ ]:
%%bash
set -euo pipefail
# Download 5 train shards + val — shared by all three models
cd /content/parameter-golf
python3 data/cached_challenge_fineweb.py --variant sp8192 --train-shards 5
echo "Data ready."

## April 9 — `SP8192_3LayerRecur_ParResid_QK525_LegalTTT`

Baseline record. Uses legacy fixed-bits export (6-bit matrices, 8-bit embeddings).
No attention gate, no multi-phase TTT.

Expected log lines: `pre-quantization post-ema`, `quantized`, `quantized_sliding_window`, `artifact bytes`.

In [ ]:
%%bash
set -euo pipefail
COLAB_DIR=/content/parameter-golf/colab/2026-04-23_SmokeTest_Records
REC=/content/parameter-golf/records/track_10min_16mb/2026-04-09_SP8192_3LayerRecur_ParResid_QK525_LegalTTT
cd "$REC"
mkdir -p logs

PYTHONPATH="$COLAB_DIR:${PYTHONPATH:-}" \
DATA_DIR=/content/parameter-golf/data \
SEED=42 \
RUN_ID=smoke_apr9 \
ITERATIONS=300 \
TRAIN_BATCH_TOKENS=32768 \
VAL_BATCH_TOKENS=32768 \
MAX_WALLCLOCK_SECONDS=0 \
VAL_LOSS_EVERY=0 \
TRAIN_LOG_EVERY=50 \
PG_COLAB_DISABLE_COMPILE=1 \
GPTQ_CALIBRATION_BATCHES=8 \
GPTQ_RESERVE_SECONDS=30 \
MATRIX_BITS=6 \
EMBED_BITS=8 \
python3 train_gpt.py

## April 20 #1 — `SP8192_AttnGate_MultiPhaseTTT_LaCT`

Adds zero-init attention output gate (`ATTN_GATE_ENABLED=1`) and 4-phase score-first TTT
(`MULTIPHASE_TTT_ENABLED=1`). LaCT adapter is disabled here to keep the run fast.
Uses the entropy-constrained GPTQ allocator.

Expected log lines: `pre-quantization post-ema`, `quantized`, `quantized_sliding_window`,
`allocator_selected`, `artifact bytes`.

In [ ]:
%%bash
set -euo pipefail
COLAB_DIR=/content/parameter-golf/colab/2026-04-23_SmokeTest_Records
REC=/content/parameter-golf/records/track_10min_16mb/2026-04-20_SP8192_AttnGate_MultiPhaseTTT_LaCT
cd "$REC"
mkdir -p logs

PYTHONPATH="$COLAB_DIR:${PYTHONPATH:-}" \
DATA_DIR=/content/parameter-golf/data \
SEED=42 \
RUN_ID=smoke_apr20_attngate_mpttt \
ITERATIONS=300 \
TRAIN_BATCH_TOKENS=32768 \
VAL_BATCH_TOKENS=32768 \
MAX_WALLCLOCK_SECONDS=0 \
VAL_LOSS_EVERY=0 \
TRAIN_LOG_EVERY=50 \
PG_COLAB_DISABLE_COMPILE=1 \
GPTQ_CALIBRATION_BATCHES=8 \
GPTQ_RESERVE_SECONDS=30 \
ATTN_GATE_ENABLED=1 \
MULTIPHASE_TTT_ENABLED=1 \
TTT_ENABLED=0 \
LACT_TTT_ENABLED=0 \
EXPORT_ALLOCATOR=entropy \
ARTIFACT_TARGET_BYTES=16000000 \
ALLOCATOR_MATRIX_BITS=5,6,7 \
ALLOCATOR_EMBED_BITS=8 \
python3 train_gpt.py

## April 20 #2 — `SP8192_3LayerRecur_ParResid_QK525_AttnGate_MP4TTT`

Combines 3-layer recurrence + parallel residual with attention output gate
(`ATTN_OUT_GATE_ENABLED=1`, note different flag name from #1) and 4-phase TTT
(`TTT_PHASES=4`). Has built-in torch SDPA fallback — no flash-attn needed.
Uses the entropy-constrained GPTQ allocator.

Expected log lines: same as April 20 #1.

In [ ]:
%%bash
set -euo pipefail
COLAB_DIR=/content/parameter-golf/colab/2026-04-23_SmokeTest_Records
REC=/content/parameter-golf/records/track_10min_16mb/2026-04-20_SP8192_3LayerRecur_ParResid_QK525_AttnGate_MP4TTT
cd "$REC"
mkdir -p logs

PYTHONPATH="$COLAB_DIR:${PYTHONPATH:-}" \
DATA_DIR=/content/parameter-golf/data \
SEED=42 \
RUN_ID=smoke_apr20_mp4ttt \
ITERATIONS=300 \
TRAIN_BATCH_TOKENS=32768 \
VAL_BATCH_TOKENS=32768 \
MAX_WALLCLOCK_SECONDS=0 \
VAL_LOSS_EVERY=0 \
TRAIN_LOG_EVERY=50 \
PG_COLAB_DISABLE_COMPILE=1 \
GPTQ_CALIBRATION_BATCHES=8 \
GPTQ_RESERVE_SECONDS=30 \
ATTN_OUT_GATE_ENABLED=1 \
MULTIPHASE_TTT_ENABLED=1 \
TTT_PHASES=4 \
TTT_ENABLED=0 \
REQUIRE_FLASH_ATTN=0 \
EXPORT_ALLOCATOR=entropy \
ARTIFACT_TARGET_BYTES=16000000 \
ALLOCATOR_MATRIX_BITS=5,6,7 \
ALLOCATOR_EMBED_BITS=8 \
python3 train_gpt.py

## Compare final metrics from all three runs

Run this after all three training cells finish.

In [ ]:
%%bash
set -euo pipefail
APR9=/content/parameter-golf/records/track_10min_16mb/2026-04-09_SP8192_3LayerRecur_ParResid_QK525_LegalTTT/logs/smoke_apr9.txt
APR20A=/content/parameter-golf/records/track_10min_16mb/2026-04-20_SP8192_AttnGate_MultiPhaseTTT_LaCT/logs/smoke_apr20_attngate_mpttt.txt
APR20B=/content/parameter-golf/records/track_10min_16mb/2026-04-20_SP8192_3LayerRecur_ParResid_QK525_AttnGate_MP4TTT/logs/smoke_apr20_mp4ttt.txt

for LOG in "$APR9" "$APR20A" "$APR20B"; do
  echo "=== $(basename $(dirname $LOG))/$(basename $LOG) ==="
  grep -E "pre-quantization post-ema|quantized|artifact bytes|allocator_selected" "$LOG" | tail -20
  echo
done